In [1]:
!pip install langchain langchain-core langchain-community duckduckgo-search langchain_experimental pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.8 MB/s eta 0:00:00


##built in tool DUCKDUCKGOSEARCH

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool=DuckDuckGoSearchRun()
results=search_tool.invoke('ipl news')
print(results)

Former Lucknow Super Giants captain Rishabh Pant is set for a return to Delhi Capitals (DC), while Kuldeep Yadav will join Lucknow Super Giants (LSG), following one of the most significant player trades in recent IPL history. Aug 7, 2026 · Follow IPL Trade 2027 news, player trade rumours, confirmed transfers, team updates, retention talks, auction buzz, and latest squad changes. Read IPL trending news, latest updates, match stories, team reports and player headlines on Today IPL. The Board of Control for Cricket in India (BCCI) is pleased to announce Phase 2 of the TATA IPL Fan Parks 2026, which will bring the electrifying IPL experience to 30 cities across 18 states and one Union Territory, further expanding the league’s footprint across the country. Get the latest cricket news, match updates, player insights & breaking stories on IPL.com. Stay ahead with real-time cricket headlines, live scores, and in-depth analysis from top leagues worldwide.


##shell tool

In [9]:
from langchain_community.tools import ShellTool
shell_tool=ShellTool()
results=shell_tool.invoke('pwd')
print(results)

Executing command:
 pwd
/content



##custom tools

In [10]:
from langchain_core.tools import tool

In [11]:
#step 1 create a function
def multiply(a,b):
  """multiply two numbers"""
  return a*b

In [12]:
#step 2 add type hints
def multiply(a:int,b:int) ->int:
  """multiply two numbers"""
  return a*b

In [13]:
#step 3 add a tool decorator
#this tol decoirator make it a special; function which interacts with the llm
@tool
def multiply(a:int,b:int) ->int:
  """multiply two numbers"""
  return a*b

In [16]:
#all the tools are the runnables
result=multiply.invoke({"a":2,"b":3})
print(result)

6


In [17]:
#attributes with the tool
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


##structured tools

In [19]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel,Field

In [21]:
class MultiplyInput(BaseModel):
  a:int=Field(required=True,description="the first number to add")
  b:int=Field(required=True,description="the second number to add")


/tmp/ipykernel_530/2061166385.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int=Field(required=True,description="the first number to add")
/tmp/ipykernel_530/2061166385.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int=Field(required=True,description="the second number to add")


In [22]:
def multiply_fun(a:int,b:int)->int: #if we dont do type hinting here then its ok
  """multiply the two numbers"""
  return a*b

In [23]:
multiply_tool=StructuredTool.from_function(
    func=multiply_fun,
    name="multiply",
    description="multiply the two numbers",
    arg_schema=MultiplyInput #defining the strict validation through pydantic class which we created
)

In [25]:
result=multiply_tool.invoke({'a':2,'b':4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

8
multiply
multiply the two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


##BaseTool

In [31]:
from langchain_core.tools import BaseTool #this si the abstract class and @tool and the strcuturedtool also inherit from this
from pydantic import BaseModel,Field
from typing import Type

In [32]:
class MultiplyInput(BaseModel):
  a:int=Field(required=True,description="the first number to add")
  b:int=Field(required=True,description="the second number to add")


/tmp/ipykernel_530/2061166385.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int=Field(required=True,description="the first number to add")
/tmp/ipykernel_530/2061166385.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int=Field(required=True,description="the second number to add")


In [33]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [34]:
multiply_tool = MultiplyTool()

In [35]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'the first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'the second number to add', 'title': 'B', 'type': 'integer'}}


##ToolKit

In [36]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [37]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [38]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
